# Self-hosted gateway - GPT text validation

Validate an **existing APIM self-hosted gateway** in another cloud or on premises. This notebook creates or updates a stable API-key access contract on its parent APIM instance, then sends **all discovery and inference traffic to the explicit self-hosted HTTPS origin**, never the managed gateway. It does not install the gateway or redeploy shared routing.

Complete [Environment Setup](citadel-environment-setup.ipynb) and backend onboarding first. Run from a machine that can reach both Azure management APIs and the self-hosted endpoint. The existing `models` API must already be associated with the named self-hosted gateway. Its gateway image must support `llm-token-limit`, `llm-content-safety`, and workload-identity authentication. Configure DNS, trusted TLS, configuration synchronization, backend connectivity, and edge-compatible authentication for **shared inference policies and the Content Safety backend** beforehand. The APIM instance's Azure managed identity is not automatically available off-cloud.

PII processing uses the repository's Language-service anonymization implementation, copied into this contract with `authentication-provider="WorkloadIdentity"` instead of the APIM instance's user-assigned identity. Configure Microsoft Entra Workload ID on the gateway workload and grant that identity access to the Language service. Existing `piiServiceUrl` named value and Content Safety backend are prerequisites. PII errors fail closed. This scenario uses **request anonymization only**: no deanonymization, state persistence, or real personal data.

References: [token limits](https://learn.microsoft.com/azure/api-management/llm-token-limit-policy), [Content Safety](https://learn.microsoft.com/azure/api-management/llm-content-safety-policy), [self-hosted workload identity](https://learn.microsoft.com/azure/api-management/authentication-managed-identity-policy#self-hosted-gateway-with-workload-identity). Token counters are local to each gateway, not a global cross-cloud quota.

Only filtered, configured GPT text/chat deployments are called, sequentially through `/models/chat/completions`. Images, embeddings, audio, realtime, non-GPT models, and aliases are excluded. Discovery uses **`GET /models/deployments` with the new contract key**. Results distinguish configured policy, successful policy traversal, and negative enforcement evidence. Requests incur provider and safety-service costs.

## 1. Configure

**Input:** The repository validation kernel. Set `self_hosted_gateway_url` to an HTTPS **origin**, without `/models`, and `self_hosted_gateway_id` to the APIM gateway resource name (not `managed`). Alternatively supply `VALIDATION_SELF_HOSTED_GATEWAY_URL` and `VALIDATION_SELF_HOSTED_GATEWAY_ID` through the process environment or selected azd environment. For private CAs, set `REQUESTS_CA_BUNDLE`; TLS verification is never disabled.

All four feature switches default to **True**. Set any to **False** before initialization to omit that feature from the new contract. Inherited global/API policies remain in effect. `model_allowlist=None` selects all eligible configured GPT text models; an explicit list narrows both the tests and RBAC grant. `skipped_models` and `operation_overrides` follow the other notebooks. Empty plans fail before provisioning.

Optional Content Safety denial evidence uses a **benign** phrase already present in an existing Content Safety text blocklist: set `content_safety_blocklist_id` and `content_safety_probe_text` together. No blocklist is created or changed. Without this fixture, negative Content Safety enforcement is explicitly SKIP, not PASS.

**Output:** Editable configuration only; no Azure resources or requests.

In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / 'azure.yaml').is_file() and (p / 'shared' / 'utils.py').is_file()), None)
if root is None:
    raise RuntimeError('Run from the repository root or validation directory.')
sys.path.insert(0, str(root / 'shared'))
import utils
with utils.notebook_step('1. Configure', inputs={'repository': str(root)}, expected='Configuration only; no cloud operations.', namespace=globals()):
    import copy
    import json
    import os
    import time
    import uuid
    import xml.etree.ElementTree as ET
    from urllib.parse import urlsplit
    import requests
    from apimtools import APIMClientTool
    from contracttools import scoped_deployment_name
    from contracttools import access_policy, prepare_artifact
    from scenariotools import policy_xml, completion_text, access_parameters, contract_identity, cleanup_products, save_manifest
    from validationtools import ValidationContext, ValidationRun

    environment = None
    from_azd = True
    overrides = {}
    operation_overrides = None
    skipped_models = None
    self_hosted_gateway_url = None
    self_hosted_gateway_id = None
    model_allowlist = None
    enable_token_limit = True
    enable_model_rbac = True
    enable_content_safety = True
    enable_pii_processing = True
    tokens_per_minute = 100000
    token_quota = 1000000
    token_quota_period = 'Hourly'
    content_safety_backend_id = 'content-safety-backend'
    content_safety_blocklist_id = None
    content_safety_probe_text = None
    sync_timeout_seconds = 90
    cleanup_enabled = False
    scenario = 'self-hosted-gateway'
    pii_fixture = 'validation.user@example.com'


## 2. Initialize and verify the edge target

**Input:** Azure coordinates from azd/overrides, explicit self-hosted origin and gateway ID, feature switches, and optional model filters.

**Output:** A nonempty filtered GPT text plan, verified gateway/API association, and a highlighted feature table. Missing settings, invalid URLs, managed-gateway targets, and unavailable enabled-feature prerequisites stop before contract provisioning. No shared API association or routing is changed.

In [ ]:
with utils.notebook_step('2. Initialize', inputs={'environment': environment, 'overrides': overrides}, expected='Verified self-hosted target, filtered GPT text models and configured features.', namespace=globals()):
    context = ValidationContext.load(overrides=overrides, environment=environment, from_azd=from_azd,
        operation_overrides=operation_overrides, skipped_models=skipped_models)
    self_hosted_gateway_url = self_hosted_gateway_url if self_hosted_gateway_url is not None else os.environ.get(
        'VALIDATION_SELF_HOSTED_GATEWAY_URL', context.azd_snapshot.get('VALIDATION_SELF_HOSTED_GATEWAY_URL'))
    self_hosted_gateway_id = self_hosted_gateway_id if self_hosted_gateway_id is not None else os.environ.get(
        'VALIDATION_SELF_HOSTED_GATEWAY_ID', context.azd_snapshot.get('VALIDATION_SELF_HOSTED_GATEWAY_ID'))
    utils.require_inputs({**{f'gateway.{k}': context.config.get(k) for k in
        ('subscription_id', 'resource_group', 'location', 'apim_name')},
        'self_hosted_gateway_url': self_hosted_gateway_url, 'self_hosted_gateway_id': self_hosted_gateway_id})
    features = {'Token limit': enable_token_limit, 'Model RBAC': enable_model_rbac,
        'Content Safety': enable_content_safety, 'PII processing': enable_pii_processing}
    if any(not isinstance(value, bool) for value in features.values()):
        raise ValueError('Feature switches must be True or False, not strings or numbers.')
    origin = urlsplit(self_hosted_gateway_url.strip())
    if (origin.scheme != 'https' or not origin.hostname or origin.username or origin.password
            or origin.path not in ('', '/') or origin.query or origin.fragment):
        raise ValueError('self_hosted_gateway_url must be an HTTPS origin without credentials, path, query or fragment.')
    self_hosted_gateway_url = self_hosted_gateway_url.strip().rstrip('/')
    self_hosted_gateway_id = self_hosted_gateway_id.strip()
    if self_hosted_gateway_id.casefold() == 'managed' or '/' in self_hosted_gateway_id:
        raise ValueError('Provide the self-hosted gateway resource name, not managed or a resource ID.')
    if not isinstance(sync_timeout_seconds, (int, float)) or isinstance(sync_timeout_seconds, bool) or not 1 <= sync_timeout_seconds <= 600:
        raise ValueError('sync_timeout_seconds must be between 1 and 600.')
    if enable_content_safety:
        utils.require_inputs({'content_safety_backend_id': content_safety_backend_id})
        if content_safety_blocklist_id is not None or content_safety_probe_text is not None:
            utils.require_inputs({'content_safety_blocklist_id': content_safety_blocklist_id,
                                  'content_safety_probe_text': content_safety_probe_text})
    catalog = context.catalog
    excluded_families = ('image', 'embedding', 'audio', 'realtime', 'transcribe', 'tts', 'whisper')
    eligible_models = sorted({entry.name for entry in catalog.entries
        if 'gpt' in entry.name.casefold() and not any(word in entry.name.casefold() for word in excluded_families)
        and 'chat' in entry.capabilities and not catalog.is_skipped(entry.name)
        and entry.backend_type in ('azure-openai', 'ai-foundry', 'aws-bedrock-mantle', 'gemini-openai')})
    if model_allowlist is not None:
        if (not isinstance(model_allowlist, list) or not model_allowlist
                or any(not isinstance(name, str) or not name.strip() for name in model_allowlist)):
            raise ValueError('model_allowlist must be a nonempty list of exact configured model names, or None.')
        invalid_models = sorted(set(model_allowlist) - set(eligible_models))
        if invalid_models:
            raise ValueError(f'model_allowlist contains unavailable, excluded or non-GPT-text models: {invalid_models}')
    selected_models = eligible_models if model_allowlist is None else sorted(set(model_allowlist))
    if not selected_models:
        raise ValueError('No eligible GPT text/chat models remain after filtering; check backend configuration, operation overrides and skipped_models.')
    apim = APIMClientTool(context.config['resource_group'], context.config['apim_name'])
    apim.initialize_context(context.account)
    if origin.hostname.casefold() == urlsplit(apim.apim_resource_gateway_url).hostname.casefold():
        raise ValueError('The self-hosted endpoint must not be the Azure managed gateway hostname.')
    coordinates = (context.config['resource_group'], apim.apim_resource_name)
    apim.client.gateway.get(*coordinates, self_hosted_gateway_id)
    models_descriptor = apim.discover_api_exact('models')
    associated_apis = {api.name for api in apim.client.gateway_api.list_by_service(*coordinates, self_hosted_gateway_id)}
    if models_descriptor['name'] not in associated_apis:
        raise ValueError('Associate the models API with the selected self-hosted gateway in APIM, then rerun initialization.')
    if enable_content_safety:
        apim.client.backend.get(*coordinates, content_safety_backend_id)
    if enable_pii_processing:
        apim.client.named_value.get(*coordinates, 'piiServiceUrl')
    preexisting_products = {product.name for product in apim.client.product.list_by_service(*coordinates)}
    owned_products = set()
    run_id = uuid.uuid4().hex[:12]
    run_dir = root / 'validation' / '.runs' / scenario / run_id
    run = ValidationRun(scenario)
    models_api = f"{self_hosted_gateway_url}/{models_descriptor['path'].strip('/')}"
    feature_rows = [{'Feature': name, 'Configured': 'ON' if enabled else 'OFF', 'Scope': 'New access contract'}
                    for name, enabled in features.items()]
    utils.format_table('ACCESS CONTRACT FEATURES', feature_rows, status='INFO')
    utils.format_output('Self-hosted target', {'gateway_id': self_hosted_gateway_id, 'endpoint': models_api,
        'selected_models': selected_models, 'token_capacity': {'tpm': tokens_per_minute, 'quota': token_quota,
        'period': token_quota_period} if enable_token_limit else 'OFF', 'pii_mode': 'Request anonymization; workload identity; fail closed' if enable_pii_processing else 'OFF'}, part='RESULT')


## 3. Build and preview the contract

**Input:** The feature switches and selected models. RBAC is a subscription/product model allowlist, not Entra user-role authentication. Only the discovered `models` API is granted.

**Output:** Editable policy XML and `.bicepparam` in this run's ignored artifact directory. Token limiting, Content Safety and PII run only on chat POSTs, not bodyless deployment discovery. A synthetic email regex makes the masking probe deterministic while still exercising the Language service. Diagnostic response headers identify the actual gateway, contract run, completed safety check, and masking transformation; they contain no credentials or PII. Disabling a switch removes its contract policy, but does not bypass inherited policies.

In [ ]:
with utils.notebook_step('3. Preview contract', inputs={'features': features, 'models': selected_models}, expected='Editable contract parameters and policy; no deployment.', namespace=globals()):
    contract = {'business_unit': 'Validation', 'use_case_name': 'SelfHosted',
        'environment': 'DEV', 'apikey_secret': 'SelfHosted-KEY', 'use_keyvault': False,
        'description': 'Self-hosted GPT text validation: ' + ', '.join(f'{name}={enabled}' for name, enabled in features.items())}
    policy_root = ET.fromstring(access_policy(selected_models if enable_model_rbac else [],
        tokens_per_minute if enable_token_limit else 1, token_quota if enable_token_limit else 1,
        token_quota_period if enable_token_limit else 'Hourly', jwt_required=False))
    inbound = policy_root.find('inbound')
    capacity = inbound.find('llm-token-limit')
    inbound.remove(capacity)
    if not enable_model_rbac:
        inbound.remove(next(node for node in inbound.findall('include-fragment')
            if node.get('fragment-id') == 'validate-model-access'))
    chat = ET.SubElement(ET.SubElement(inbound, 'choose'), 'when',
        {'condition': '@(context.Request.Method == "POST" && context.Request.Url.Path.EndsWith("/chat/completions"))'})
    if enable_token_limit:
        capacity.set('remaining-quota-tokens-header-name', 'x-citadel-capacity-remaining')
        chat.append(capacity)
    if enable_content_safety:
        safety = ET.SubElement(chat, 'llm-content-safety', {'backend-id': content_safety_backend_id,
            'shield-prompt': 'false', 'enforce-on-completions': 'false'})
        categories = ET.SubElement(safety, 'categories', {'output-type': 'EightSeverityLevels'})
        for category in ('Hate', 'SelfHarm', 'Sexual', 'Violence'):
            ET.SubElement(categories, 'category', {'name': category, 'threshold': '4'})
        if content_safety_blocklist_id:
            ET.SubElement(ET.SubElement(safety, 'blocklists'), 'id').text = content_safety_blocklist_id
        ET.SubElement(chat, 'set-variable', {'name': 'selfHostedSafetyChecked', 'value': '@(true)'})
    if enable_pii_processing:
        ET.SubElement(chat, 'set-variable', {'name': 'piiAnonymizationEnabled', 'value': 'true'})
        ET.SubElement(chat, 'set-variable', {'name': 'continueOnNLPError', 'value': 'false'})
        ET.SubElement(chat, 'set-variable', {'name': 'piiRegexPatterns', 'value': json.dumps([
            {'pattern': r'validation\.user@example\.com', 'category': 'VALIDATION_EMAIL'}])})
        ET.SubElement(chat, 'set-variable', {'name': 'selfHostedOriginalContent',
            'value': '@(context.Request.Body.As<string>(preserveContent: true))'})
        ET.SubElement(chat, 'set-variable', {'name': 'piiInputContent',
            'value': '@(context.Variables.GetValueOrDefault<string>("selfHostedOriginalContent"))'})
        pii_source = root / 'bicep' / 'infra' / 'modules' / 'apim' / 'policies' / 'frag-pii-anonymization.xml'
        pii_fragment = ET.fromstring(policy_xml(pii_source.read_text(encoding='utf-8')))
        for parent in pii_fragment.iter():
            for authentication in parent.findall('authentication-managed-identity'):
                parent.remove(authentication)
        for service_request in pii_fragment.iter('send-request'):
            for header in service_request.findall('set-header'):
                if header.get('name', '').casefold() == 'authorization':
                    service_request.remove(header)
            ET.SubElement(service_request, 'authentication-managed-identity',
                {'resource': 'https://cognitiveservices.azure.com',
                 'authentication-provider': 'WorkloadIdentity', 'ignore-error': 'false'})
        chat.extend(copy.deepcopy(list(pii_fragment)))
        ET.SubElement(chat, 'set-variable', {'name': 'selfHostedPiiMasked', 'value':
            '@(context.Variables.GetValueOrDefault<string>("selfHostedOriginalContent") != context.Variables.GetValueOrDefault<string>("piiAnonymizedContent"))'})
        ET.SubElement(chat, 'set-body').text = '@(context.Variables.GetValueOrDefault<string>("piiAnonymizedContent"))'
    outbound = policy_root.find('outbound')
    diagnostic_headers = {'x-citadel-gateway-id': '@(context.Deployment.Gateway.Id)', 'x-citadel-contract-run': run_id}
    if enable_content_safety:
        diagnostic_headers['x-citadel-safety-checked'] = '@(context.Variables.GetValueOrDefault<bool>("selfHostedSafetyChecked", false).ToString().ToLowerInvariant())'
    if enable_pii_processing:
        diagnostic_headers['x-citadel-pii-masked'] = '@(context.Variables.GetValueOrDefault<bool>("selfHostedPiiMasked", false).ToString().ToLowerInvariant())'
    for name, value in diagnostic_headers.items():
        ET.SubElement(ET.SubElement(outbound, 'set-header', {'name': name, 'exists-action': 'override'}), 'value').text = value
    contract_policy = ET.tostring(policy_root, encoding='unicode')
    parameters = access_parameters(context, apim, contract, contract_policy, api_names=[models_descriptor['name']])
    parameters['globalGatewayUrl'] = self_hosted_gateway_url
    artifact = prepare_artifact(root / 'bicep' / 'infra' / 'citadel-access-contracts' / 'main.bicep',
        parameters, run_dir / 'access', policies={'self-hosted-policy.xml': contract_policy})
    utils.format_output('Contract preview', {'features': feature_rows, 'policy_xml': contract_policy,
        'parameters': artifact.params_file.read_text(encoding='utf-8')}, part='RESULT')


## 4. Deploy and retrieve the exact subscription credential

**Input:** The generated `.bicepparam` and policy XML. Azure CLI authentication and subscription-scope deployment permissions are required.

**Output:** A stable product/subscription granting only the `models` API, with its exact key fetched afresh and kept only in memory. The deployed policy is compared with the preview, so manual edits require updating the notebook configuration and rerunning. Ownership is recorded only for a product absent before this run. Reused products, shared APIs, backends, identities and gateway associations are retained.

In [ ]:
with utils.notebook_step('4. Deploy contract', inputs={'parameters': str(artifact.params_file)}, expected='Deployed contract, verified policy and in-memory credential.', namespace=globals()):
    outputs = artifact.deploy(context.config['location'], scoped_deployment_name('validation-self-hosted', apim.apim_service_id),
        required_outputs=('products', 'subscriptions'))
    product_id, subscription_name = contract_identity(outputs, preexisting_products, owned_products)
    deployed_policy = apim.client.product_policy.get(*coordinates, product_id, 'policy').value
    expected_policy = ET.canonicalize(contract_policy, strip_text=True)
    actual_policy = ET.canonicalize(deployed_policy, strip_text=True)
    run.check('deployed-contract-policy', actual_policy == expected_policy,
        expected='Deployed policy matches configured feature switches and model grants', actual=actual_policy == expected_policy)
    if actual_policy != expected_policy:
        raise RuntimeError('Deployed policy differs from this notebook configuration; reinitialize and redeploy.')
    api_key = apim.get_subscription_key(subscription_name)
    headers = {'api-key': api_key, 'Content-Type': 'application/json'}
    manifest_path = save_manifest(run_dir, context, apim, owned_products)
    utils.format_table('DEPLOYED ACCESS CONTRACT FEATURES', feature_rows, status='PASS')
    utils.format_output('Contract deployed', {'product_id': product_id, 'subscription_name': subscription_name,
        'self_hosted_endpoint': models_api, 'selected_models': selected_models}, part='RESULT')


## 5. Discover deployments through the new contract

**Input:** `GET /models/deployments` on the self-hosted endpoint, authenticated with the exact new subscription key.

**Output:** The actual contract-visible deployment names and the resulting test plan. While a new subscription synchronizes, only HTTP 401/403/404 are retried with visible status, for the bounded synchronization window. Other HTTP, transport, TLS or malformed-response errors fail immediately. Redirects are never followed. With RBAC ON, discovery must equal the selected allowlist; with RBAC OFF, additional deployments may be visible but are not called. The managed endpoint is never a fallback.

In [ ]:
with utils.notebook_step('5. Discover /models/deployments', inputs={'endpoint': models_api + '/deployments'}, expected='Contract-visible deployments and a nonempty GPT text plan.', namespace=globals()):
    deadline = time.monotonic() + sync_timeout_seconds
    with requests.Session() as session:
        session.headers.update(headers)
        while True:
            response = utils.http_call(session.get, models_api + '/deployments', timeout=30, allow_redirects=False)
            if response.status_code == 200:
                break
            if response.status_code not in (401, 403, 404) or time.monotonic() >= deadline:
                raise RuntimeError(f'Self-hosted deployment discovery failed: HTTP {response.status_code}. Check association, synchronization, authentication and policy support.')
            utils.format_output('Waiting for edge configuration sync', {'HTTP': response.status_code,
                'gateway': self_hosted_gateway_id}, status='WARNING', part='RESULT')
            time.sleep(min(5, max(0, deadline - time.monotonic())))
    body = response.json()
    deployments = body.get('value') if isinstance(body, dict) else None
    if not isinstance(deployments, list) or not deployments:
        raise ValueError('Expected a nonempty value array from /models/deployments.')
    if any(not isinstance(item, dict) or not isinstance(item.get('name'), str) or not item['name'].strip() for item in deployments):
        raise ValueError('Every deployment must have a nonempty name; backend IDs are not model names.')
    discovered_models = sorted({item['name'] for item in deployments})
    visible = {name.casefold() for name in discovered_models}
    selected = {name.casefold() for name in selected_models}
    run.check('selected-deployments-visible', selected.issubset(visible), expected=selected_models, actual=discovered_models)
    if enable_model_rbac:
        run.check('discovery-model-rbac', visible == selected, expected=selected_models, actual=discovered_models)
    test_models = [name for name in selected_models if name.casefold() in visible]
    utils.format_output('Self-hosted deployment inventory', {'available_through_contract': discovered_models,
        'filtered_gpt_text_loop': test_models, 'features': feature_rows}, part='RESULT')
    if not test_models or not selected.issubset(visible) or (enable_model_rbac and visible != selected):
        raise RuntimeError('Discovery does not match the configured contract; no incomplete or overbroad model loop will run.')


## 6. Execute the filtered GPT text loop

**Input:** One short, nonstreaming chat request per discovered/selected GPT text model. The only PII is a synthetic email fixture. No temperature or version-specific completion-length parameters are imposed.

**Output:** Per-model text and usage, with required assertions for nonempty content, the actual self-hosted gateway ID and current contract run. When enabled, require the configured remaining-quota header, successful Content Safety traversal, and a real request-body masking transformation. A successful HTTP response alone does not pass these feature checks. No client retries conceal failures. Processing continues to collect other models' failures before the strict summary.

In [ ]:
with utils.notebook_step('6. GPT text loop', inputs={'models': test_models, 'features': features}, expected='Text, self-hosted identity, current policy and enabled-feature evidence for every model.', namespace=globals()):
    prompt = 'What is 2+2? Answer briefly.'
    if enable_pii_processing:
        prompt += f' Synthetic contact for masking validation: {pii_fixture}. Do not repeat the contact.'
    with requests.Session() as session:
        session.headers.update(headers)
        for model in test_models:
            try:
                response = utils.http_call(session.post, models_api + '/chat/completions',
                    json={'model': model, 'messages': [{'role': 'user', 'content': prompt}], 'stream': False},
                    timeout=120, allow_redirects=False)
                response.raise_for_status()
                if response.status_code != 200:
                    raise ValueError(f'Expected HTTP 200 without redirects; received {response.status_code}.')
                data = response.json()
                content = completion_text(data)
                run.check(model + ':chat', bool(content.strip()), expected='Nonempty text completion', actual=content)
                run.check(model + ':edge-identity', response.headers.get('x-citadel-gateway-id') == self_hosted_gateway_id,
                    expected=self_hosted_gateway_id, actual=response.headers.get('x-citadel-gateway-id'))
                run.check(model + ':contract-sync', response.headers.get('x-citadel-contract-run') == run_id,
                    expected=run_id, actual=response.headers.get('x-citadel-contract-run'))
                if enable_token_limit:
                    remaining = response.headers.get('x-citadel-capacity-remaining', '')
                    run.check(model + ':token-accounting', remaining.isdecimal() and 0 <= int(remaining) <= token_quota,
                        expected='Remaining quota header between zero and configured quota', actual=remaining)
                if enable_content_safety:
                    run.check(model + ':content-safety-traversed', response.headers.get('x-citadel-safety-checked') == 'true',
                        expected='Request completed Content Safety policy', actual=response.headers.get('x-citadel-safety-checked'))
                if enable_pii_processing:
                    run.check(model + ':pii-masked', response.headers.get('x-citadel-pii-masked') == 'true',
                        expected='Synthetic PII changed by request anonymization before inference', actual=response.headers.get('x-citadel-pii-masked'))
                utils.format_output('GPT text result', {'model': model, 'text': content,
                    'usage': data.get('usage'), 'features': feature_rows}, part='RESULT')
            except (requests.RequestException, ValueError) as exc:
                run.check(model + ':request', False, expected='Successful self-hosted text request with valid body', actual=str(exc))


## 7. Check denial behavior and coverage boundaries

**Input:** The same access-contract key. Model RBAC uses a deliberately ungranted, nonexistent GPT name and must return the repository's exact `unauthorized_model_access` code, not a generic 403/404. Optional Content Safety uses only the configured benign blocklist phrase.

**Output:** Expected RBAC denial, optional Content Safety block, and explicit skips for disabled features or missing negative fixtures. Token accounting is tested, but deliberate quota exhaustion/load testing is out of scope and reported as SKIP. With safety enabled, the blocklist test also requires a safety-specific error body; authentication or RBAC errors cannot count as a safety block.

In [ ]:
with utils.notebook_step('7. Denial checks', inputs={'model_rbac': enable_model_rbac, 'content_safety': enable_content_safety}, expected='Specific policy denials or explicit optional skips.', namespace=globals()):
    for name, enabled in features.items():
        if not enabled:
            run.skip(name, 'Explicitly disabled on this access contract; inherited policies are unchanged.')
    with requests.Session() as session:
        session.headers.update(headers)
        if enable_model_rbac:
            denied_model = 'gpt-ungranted-' + run_id
            try:
                response = utils.http_call(session.post, models_api + '/chat/completions',
                    json={'model': denied_model, 'messages': [{'role': 'user', 'content': 'What is 2+2?'}]},
                    timeout=30, allow_redirects=False)
                data = response.json()
                error = data.get('error') if isinstance(data, dict) else None
                code = error.get('code') if isinstance(error, dict) else None
                run.check('model-rbac-denial', response.status_code == 403 and code == 'unauthorized_model_access',
                    expected='403 unauthorized_model_access', actual={'status': response.status_code, 'code': code})
            except (requests.RequestException, ValueError) as exc:
                run.check('model-rbac-denial', False, expected='403 unauthorized_model_access', actual=str(exc))
        if enable_content_safety and content_safety_blocklist_id:
            try:
                response = utils.http_call(session.post, models_api + '/chat/completions',
                    json={'model': test_models[0], 'messages': [{'role': 'user', 'content': content_safety_probe_text}]},
                    timeout=60, allow_redirects=False)
                data = response.json()
                error = data.get('error', data) if isinstance(data, dict) else {}
                signature = json.dumps(error).casefold()
                safety_error = any(term in signature for term in ('content safety', 'content_safety', 'contentsafety', 'blocklist'))
                run.check('content-safety-blocklist-denial', response.status_code == 403 and safety_error,
                    expected='403 with Content Safety/blocklist-specific error', actual={'status': response.status_code, 'error': error})
            except (requests.RequestException, ValueError) as exc:
                run.check('content-safety-blocklist-denial', False, expected='Content Safety block', actual=str(exc))
        elif enable_content_safety:
            run.skip('content-safety-negative-enforcement', 'No benign blocklist fixture supplied; only successful safety-policy traversal was verified.')
    if enable_token_limit:
        run.skip('token-limit-exhaustion', 'Remaining-quota accounting is checked per model; deliberate quota exhaustion is not performed by this smoke test.')


## 8. Strict summary

**Input:** All deployment, discovery, per-model and feature results.

**Output:** Highlighted configured contract features, target, tested models, and a strict PASS/FAIL/SKIP table. Any required failure raises; optional untested enforcement is never presented as PASS. Artifacts remain available for investigation.

In [ ]:
with utils.notebook_step('8. Summary', inputs={'gateway': self_hosted_gateway_id, 'product': product_id}, expected='Strict verdict with configured features and explicit coverage limits.', namespace=globals()):
    utils.format_table('CONFIGURED ACCESS CONTRACT FEATURES', feature_rows, status='INFO')
    utils.format_output('Self-hosted validation scope', {'endpoint': models_api, 'gateway_id': self_hosted_gateway_id,
        'product': product_id, 'tested_gpt_models': test_models, 'manifest': str(manifest_path)}, part='RESULT')
    summary = run.finish()


## 9. Optional cleanup

**Input:** `cleanup_enabled=False` by default and positively recorded product ownership. Run this cell manually even if an earlier test failed.

**Output:** A preview of this run's owned product, or its deletion (including its subscription) when explicitly enabled. The self-hosted gateway, API association, shared policies/backends and preexisting products are never deleted.

In [ ]:
with utils.notebook_step('9. Optional cleanup', inputs={'enabled': cleanup_enabled, 'owned_products': sorted(owned_products)}, expected='Ownership-scoped cleanup preview or explicit deletion.', namespace=globals()):
    cleanup_products(apim, owned_products, enabled=cleanup_enabled)
    manifest_path = save_manifest(run_dir, context, apim, owned_products)
